# Haya Skin Analyzer — YOLOv8 Training
Run on Google Colab with T4 GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Cell 1 — Config
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY_HERE"  # paste your key from roboflow.com → Settings
EPOCHS = 80
MODEL_SIZE = "yolov8n"
PROJECT_NAME = "haya_skin"
RUN_NAME = "v1"

In [ ]:
# Cell 2 — Install dependencies
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

In [ ]:
# Cell 3 — Download dataset from Roboflow Universe
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# These are real public datasets on Roboflow Universe (verified June 2026)
# We try each one until one downloads successfully
datasets_to_try = [
    ("roboflow-universe-projects", "acne-detection-nzmkp",        2),
    ("roboflow-universe-projects", "acne-detection-nzmkp",        1),
    ("akash-nair",                 "acne-vulgaris",                1),
    ("roboflow-universe-projects", "skin-lesion-detection-uquig", 1),
    ("roboflow-universe-projects", "acne-jxneu",                  1),
]

DATA_YAML = None
for workspace, project_name, version in datasets_to_try:
    try:
        print(f"Trying {workspace}/{project_name} v{version}...")
        project = rf.workspace(workspace).project(project_name)
        dataset = project.version(version).download("yolov8")
        DATA_YAML = dataset.location + "/data.yaml"
        print(f"SUCCESS: {DATA_YAML}")
        break
    except Exception as e:
        print(f"  Failed: {e}")

if DATA_YAML is None:
    raise RuntimeError(
        "\n\nAll dataset downloads failed. Do this manually:\n"
        "1. Go to https://universe.roboflow.com\n"
        "2. Search: acne detection\n"
        "3. Open any dataset → Versions → Export → YOLOv8 format → Get Snippet\n"
        "4. Paste the rf.workspace(...).project(...) lines here and re-run this cell\n"
    )
else:
    print(f"\nDataset ready. data.yaml = {DATA_YAML}")

In [ ]:
# Cell 4 — Train
from ultralytics import YOLO

model = YOLO(f"{MODEL_SIZE}.pt")

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    patience=25,
    save=True,
    project=PROJECT_NAME,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
)

print(f"Training complete. Best model: {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")

In [ ]:
# Cell 5 — Evaluate
best_model = YOLO(f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
metrics = best_model.val(data=DATA_YAML)

print(f"\nmAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print("\nTarget: mAP50 > 0.60 = good, > 0.75 = excellent")

In [ ]:
# Cell 6 — Download best.pt
from google.colab import files
files.download(f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
print("Download started.")
print("Save the file to: backend/app/models/skin_model.pt")